# Lecture 2 — Interactive Maps, Modern Raster & Big Geo Data
## Session 2 · Geo Agents

**Anzony Quispe · Jesus Gastañaduy · 2025**

---

### What we cover

| Part | Topic | Libraries |
|------|-------|-----------|
| 1 | Interactive maps | `folium`, `lonboard` |
| 2 | Modern raster | `geowombat` |
| 3 | Spatial SQL & scale | `GeoParquet`, `DuckDB`, `dask-geopandas` |
| 4 | The modern geo stack | (recap) |
| 5 | Replication papers | 10 policy-relevant papers |

> This lecture explains **what each package is for**. The companion **`Lecture2_Tutorial.ipynb`** integrates all of them in a single Peru case study.

---
## 0 · Setup

All packages are declared in `pyproject.toml`. If any import fails, see `SETUP.md`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# Interactive maps
import folium
# import lonboard              # uncomment when installed
# from lonboard import Map, ScatterplotLayer

# Modern raster
# import geowombat as gw       # uncomment when installed

# Spatial SQL & big data
import duckdb
import dask_geopandas as dgpd
import pyarrow

print('pandas         :', pd.__version__)
print('geopandas      :', gpd.__version__)
print('duckdb         :', duckdb.__version__)
print('pyarrow        :', pyarrow.__version__)
print('folium         :', folium.__version__)

---
## Part 1 — Interactive Maps

### Static vs interactive: two different jobs

| | **Static** (matplotlib) | **Interactive** (folium, lonboard) |
|---|---|---|
| Reproducible in a paper | ✅ | ❌ (needs browser) |
| Pan / zoom / click | ❌ | ✅ |
| Layer toggles | ❌ | ✅ |
| Shareable HTML | ❌ | ✅ |
| Handles millions of points | ✅ (as a heatmap) | ✅ (with `lonboard`) |

**Rule of thumb:** use matplotlib for a manuscript, `folium`/`lonboard` for a dashboard or a stakeholder meeting.

### 1.1  `folium` — Python bindings to Leaflet.js

**What it is:** a thin Python wrapper around **Leaflet.js**, the classic JS
map library. Every folium `Map` renders to a self-contained HTML file that
you can email, host on GitHub Pages, or embed in a Streamlit app.

**Best for:**
- Fewer than ~10 000 features
- Choropleths of administrative boundaries
- Public-facing HTML deliverables

In [ ]:
# Load Peru regions (Level 1) — same file as Lecture 1
peru = gpd.read_file(
    'https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_PER_1.json'
)
peru = peru[['GID_1', 'NAME_1', 'geometry']].copy()

# Base map centred on Peru, using CartoDB Positron basemap
m = folium.Map(
    location=[-9.19, -75.02],
    zoom_start=6,
    tiles='CartoDB positron'
)

# Add a simple GeoJson layer with hover popups
folium.GeoJson(
    peru,
    name='Regions',
    tooltip=folium.GeoJsonTooltip(fields=['NAME_1'], aliases=['Region:']),
    style_function=lambda x: {
        'fillColor': '#3b82f6',
        'color'    : '#1d4ed8',
        'weight'   : 1,
        'fillOpacity': 0.3,
    }
).add_to(m)

folium.LayerControl().add_to(m)
m   # Jupyter renders inline; save with m.save('peru.html')

### 1.2  A folium choropleth in 5 lines

The `folium.Choropleth` helper does the boundary lookup and colour
scaling for you. All it needs is:
- A GeoJson layer with a **key column**
- A dataframe with the same key and a numeric value column

In [ ]:
# Fake data — 'population' per region
peru['population'] = np.random.randint(200_000, 10_000_000, size=len(peru))

m2 = folium.Map(location=[-9.19, -75.02], zoom_start=5, tiles='CartoDB positron')

folium.Choropleth(
    geo_data      = peru.__geo_interface__,
    data          = peru,
    columns       = ['GID_1', 'population'],
    key_on        = 'feature.properties.GID_1',
    fill_color    = 'YlOrRd',
    fill_opacity  = 0.7,
    line_opacity  = 0.4,
    legend_name   = 'Population',
).add_to(m2)

m2

### 1.3  `lonboard` — deck.gl (GPU) in Python

**What it is:** bindings to Uber's **deck.gl** library, which uses WebGL2/WebGPU
for GPU-accelerated rendering. Data flows between Python and the browser as
**GeoArrow** binary buffers — no JSON serialisation, no string parsing.

**Best for:**
- Datasets with **more than 100 000 features**
- 3D extrusion (buildings, hexagon bins, arcs)
- Speed-critical dashboards where folium starts to lag

**How it differs from folium:**

| | folium | lonboard |
|---|---|---|
| Backend | Leaflet.js (SVG/Canvas) | deck.gl (WebGL/WebGPU) |
| Data transfer | JSON | GeoArrow (binary) |
| Point capacity | ~10 000 | 10 000 000+ |
| Output | HTML file | Jupyter widget |
| Learning curve | Very shallow | Shallow |

In [ ]:
# NOTE: run only if lonboard is installed
# from lonboard import Map, ScatterplotLayer
# import numpy as np
#
# # 100 000 random points in Peru bounding box
# xs = np.random.uniform(-81, -68, size=100_000)
# ys = np.random.uniform(-18, 0,   size=100_000)
# many_pts = gpd.GeoDataFrame(
#     {'value': np.random.rand(len(xs))},
#     geometry=gpd.points_from_xy(xs, ys),
#     crs='EPSG:4326'
# )
#
# layer = ScatterplotLayer.from_geopandas(
#     many_pts,
#     get_fill_color=[255, 100, 0],
#     get_radius=500,          # in metres
#     radius_units='meters',
# )
# Map(layer, view_state={'longitude': -75, 'latitude': -9, 'zoom': 5})

### 1.4  Which one should I use?

| If your dataset is… | Use |
|---------------------|-----|
| A few polygons or points, needs to be a shareable HTML file | **folium** |
| A choropleth of admin boundaries | **folium** |
| Millions of GPS pings / taxi trips / fire detections | **lonboard** |
| Needs 3D extrusion | **lonboard** |
| The audience is on an old browser (no WebGL2) | **folium** |

> **Teaching plan:** we demo folium first because it's beginner-friendly,
> then swap in lonboard for the big-data example in the tutorial.

---
## Part 2 — Modern Raster with `geowombat`

### Where geowombat fits in

| Level | Library | You write… |
|-------|---------|------------|
| Low-level | `rasterio` | `open`, `read`, `write` with pixel windows |
| Labeled | `xarray` + `rioxarray` | Named dims, but manual for satellite bands |
| **High-level** | **`geowombat`** | 1-line NDVI, tasseled cap, cloud masking |

`geowombat` sits on top of **both** rasterio and xarray. It ships helpers
for common remote-sensing operations and streams data lazily via `dask`.

**Features you get for free:**
- Read a Landsat / Sentinel scene with band names attached
- Chunked (out-of-core) computation via dask
- Band math: `.norm_diff()`, `.tasseled_cap()`, `.evi()`
- Automatic reprojection and clipping
- Cloud-native (COG, S3, GCS)

In [ ]:
# --- Conceptual — full data would be several GB ---
# import geowombat as gw
#
# # Landsat 8 has red = B4, NIR = B5
# with gw.open(
#     ['B4.TIF', 'B5.TIF'],
#     band_names=['red', 'nir'],
#     chunks=1024,          # process in 1024x1024 tiles
# ) as src:
#     ndvi = src.gw.norm_diff('nir', 'red')      # (nir-red)/(nir+red)
#     ndvi.gw.save('ndvi_peru.tif')              # writes as COG
#
# # RAM stays flat — dask streams chunks through
# print('NDVI written to disk without ever loading the full scene.')

### 2.1  Cloud-Optimized GeoTIFF (COG)

A **COG** is a regular GeoTIFF with two important internal features:

1. **Tiled** internally (typically 512×512 blocks)
2. **Overviews** — pre-computed downsampled copies inside the same file

Because the file has an internal index, an HTTP range request can fetch
**just** the window you need — no full download.

This is the technology behind:
- **AWS Open Data** — Landsat, Sentinel-2, MODIS
- **Google Earth Engine** exports
- **Microsoft Planetary Computer**

> With `geowombat` + a URL, you can process a 10 GB scene by streaming just
> the ~100 MB you actually need.

In [ ]:
# --- Conceptual — needs internet + geowombat installed ---
# import geowombat as gw
#
# # Landsat 8 red band on AWS S3 (example URL)
# url = ('https://landsatlook.usgs.gov/data/collection02/level-2/'
#        'standard/oli-tirs/2024/006/067/LC08_L2SP_006067_20240210_...')
#
# # Only the requested window is downloaded
# with gw.open(url) as src:
#     lima_box = src.gw.subset(
#         left=-77.15, right=-77.00,
#         top=-12.00, bottom=-12.15,
#         mask_corners=False,
#     )
#     lima_box.gw.save('lima_red.tif')
# print('Read a 100 km window from a 10 GB scene — no full download.')

---
## Part 3 — Spatial SQL & Big Geo Data

When does geopandas stop being enough?

| Problem | Solution |
|---------|----------|
| File is larger than RAM | **GeoParquet** (columnar, compressed) |
| Need one small area of a huge dataset | **DuckDB spatial** (predicate pushdown) |
| Millions of joins / repeated queries | **DuckDB spatial** |
| CPU-parallel spatial join on a big dataset | **dask-geopandas** |

These three tools are the **modern big-geo stack**, used by projects like
Overture Maps, Microsoft Buildings, and Meta's HDX.

### 3.1  GeoParquet — 10× smaller than shapefile

**What it is:** Apache Parquet (the columnar file format used by Spark,
DuckDB, BigQuery) extended with a **geometry column** encoded as WKB.

**Why it wins:**
- **Compression**: ~10× smaller than shapefile
- **Column pruning**: read only the columns you need
- **Predicate pushdown**: read only rows matching a filter
- **Cloud-friendly**: partial reads via HTTP range requests
- **Zero-copy** to Arrow → lonboard, DuckDB, pandas 2.x, polars

In [ ]:
# Write and read with a one-line API
peru.to_parquet('/tmp/peru.parquet', compression='snappy')

# Read back — pandas 2.x + Arrow backend under the hood
peru2 = gpd.read_parquet('/tmp/peru.parquet')
print(f'Peru regions round-tripped: {len(peru2)} rows')
print(f'CRS preserved: {peru2.crs}')

# Read only a subset of columns
peru_geom_only = gpd.read_parquet('/tmp/peru.parquet', columns=['NAME_1', 'geometry'])
peru_geom_only.head(3)

### 3.2  DuckDB spatial — SQL on geo, zero setup

**What it is:** an in-process OLAP database (like SQLite, but columnar).
The `spatial` extension adds PostGIS-like functions: `ST_Within`,
`ST_Intersects`, `ST_Distance`, `ST_Buffer`, etc.

**Why it wins over PostGIS:**
- **No server** — runs inside your Python process
- **Reads Parquet directly** without copying it into RAM
- **Multi-threaded, vectorised** — often faster than PostGIS for OLAP queries
- **Runs in a notebook** — perfect for teaching and exploration

In [ ]:
# Fresh in-memory database
con = duckdb.connect()

# Install once, load per session
con.execute('INSTALL spatial;  LOAD spatial;')

# --- Read the parquet file we just wrote, in SQL ---
result = con.execute('''
    SELECT
        NAME_1,
        ST_Area(geometry) AS area_deg2
    FROM read_parquet('/tmp/peru.parquet')
    ORDER BY area_deg2 DESC
    LIMIT 5;
''').fetchdf()

print('Top 5 Peru regions by area (deg²):')
result

In [ ]:
# --- Spatial join in SQL ---
# Load a few fake fire points as an example
import numpy as np
np.random.seed(0)

fires = gpd.GeoDataFrame(
    {'frp': np.random.rand(5000) * 100},
    geometry = gpd.points_from_xy(
        np.random.uniform(-81, -68, 5000),
        np.random.uniform(-18, 0,   5000),
    ),
    crs='EPSG:4326'
)
fires.to_parquet('/tmp/fires.parquet')

# Fires per region, using ST_Within
q = '''
    SELECT
        p.NAME_1,
        COUNT(*)   AS n_fires,
        SUM(f.frp) AS total_frp
    FROM read_parquet('/tmp/fires.parquet') AS f
    JOIN read_parquet('/tmp/peru.parquet') AS p
      ON ST_Within(f.geometry, p.geometry)
    GROUP BY p.NAME_1
    ORDER BY n_fires DESC
    LIMIT 10;
'''
con.execute(q).fetchdf()

### 3.3  `dask-geopandas` — parallel geopandas

**What it is:** a drop-in `GeoDataFrame` that partitions data across CPU
cores. Operations are **lazy** — they build a computation graph that
executes only when you call `.compute()`.

**Best for:**
- Multi-year fires (5+ GB)
- Country-scale OSM road networks
- Repeated `sjoin` on the same base layers

**Trick:** `.spatial_shuffle()` uses a **Hilbert curve** to partition the
data spatially, so each core works on a contiguous region — this makes
spatial joins ~10× faster.

In [ ]:
# Convert an in-memory GeoDataFrame into a dask-partitioned one
fires_dgpd = dgpd.from_geopandas(fires, npartitions=4)

print(f'Partitions: {fires_dgpd.npartitions}')
print(f'Type      : {type(fires_dgpd).__name__}')

# Lazy op — nothing computed yet
mean_frp = fires_dgpd.frp.mean()
print('Before compute:', mean_frp)

# Trigger execution
print(f'After compute : mean FRP = {mean_frp.compute():.2f}')

In [ ]:
# A parallel spatial join — same API as geopandas
peru_dgpd = dgpd.from_geopandas(peru, npartitions=1)

joined = fires_dgpd.sjoin(peru_dgpd, predicate='within')

# Fires per region, computed in parallel
result = (joined.groupby('NAME_1')
                 .size()
                 .compute()
                 .sort_values(ascending=False)
                 .head(10))

print('Fires per region (via dask):')
result

---
## Part 4 — The Modern Geo Stack

Everything we saw fits into three layers:

```
                   ┌─────────────┬─────────────┬─────────────┐
    STORAGE        │ GeoParquet  │  COG (tif)  │  OSM/GADM   │
                   └──────┬──────┴──────┬──────┴──────┬──────┘
                          │             │             │
                   ┌──────▼──────┬──────▼──────┬──────▼──────┐
    COMPUTE        │   DuckDB    │dask-geopandas│  geowombat  │
                   │ (SQL, fast) │  (parallel)  │  (raster)   │
                   └──────┬──────┴──────┬──────┴──────┬──────┘
                          │             │             │
                   ┌──────▼──────┬──────▼──────┬──────▼──────┐
    PRESENT        │   folium    │   lonboard  │ matplotlib  │
                   │ (Leaflet)   │  (deck.gl)  │  (static)   │
                   └─────────────┴─────────────┴─────────────┘
```

**Key insight:** each row is swappable. Same data flows through
*storage → compute → presentation*. The tutorial notebook shows this
in one continuous pipeline.

---
## Part 5 — Ten replication papers

Every paper below uses **open data** and methods that translate cleanly
to the tools in this lecture (`geopandas`, `rasterio`, `xarray`,
`rasterstats`, `duckdb`, `osmnx`, `networkx`).

### El Niño & Peru

| Paper | Journal | Data | Method | DOI |
|-------|---------|------|--------|-----|
| **Aragón & Rud** (2013) — *Yanacocha gold mine income effects* | AEJ:Policy | Mine coord + GADM Peru + ENAHO | Distance-buffer DiD | [10.1257/pol.5.2.1](https://doi.org/10.1257/pol.5.2.1) |
| **Hsiang, Meng & Cane** (2011) — *ENSO and civil conflict* | Nature | NOAA ENSO + UCDP/PRIO | Raster-to-country panel | [10.1038/nature10311](https://doi.org/10.1038/nature10311) |
| **Caramanica et al.** (2020) — *Pre-Columbian El Niño resilience in Peru* | PNAS | Remote sensing of ancient canals | Landscape archaeology | [10.1073/pnas.2006519117](https://doi.org/10.1073/pnas.2006519117) |

### Deforestation

| Paper | Journal | Data | Method | DOI |
|-------|---------|------|--------|-----|
| **Assunção, Gandour & Rocha** (2020) — *Rural credit → Amazon deforestation* | Economic Journal | INPE PRODES + IBGE municipal | Zonal stats + DiD | [10.1093/ej/uez060](https://doi.org/10.1093/ej/uez060) |
| **Baragwanath & Bayi** (2020) — *Indigenous territories reduce loss* | PNAS | Hansen GFC + FUNAI | Spatial RD at borders | [10.1073/pnas.1917874117](https://doi.org/10.1073/pnas.1917874117) |
| **Alix-Garcia, Sims & Yañez-Pagans** (2015) — *Mexico PES program* | AEJ:Policy | MODIS/Landsat + CONAFOR | Matched DiD + zonal stats | [10.1257/pol.20130139](https://doi.org/10.1257/pol.20130139) |

### Transportation & highways

| Paper | Journal | Data | Method | DOI |
|-------|---------|------|--------|-----|
| **Tsivanidis** (2025) — *Bogotá TransMilenio BRT* | AER | Census + GTFS + OSM | Accessibility + spatial GE | openICPSR 237317 |
| **Faber** (2014) — *China National Trunk Highways* | RESTUD | Digitized NTHS + counties | Least-cost path IV | [10.1093/restud/rdu010](https://doi.org/10.1093/restud/rdu010) |
| **Storeygard** (2016) — *Africa transport costs* | RESTUD | DMSP nightlights + roads | Shortest-path + panel | [10.1093/restud/rdw020](https://doi.org/10.1093/restud/rdw020) |

### Hospitals & markets

| Paper | Journal | Data | Method | DOI |
|-------|---------|------|--------|-----|
| **Weiss, Nelson et al.** (2020) — *Global healthcare access* | Nature Medicine | OSM + land cover + DEM | Friction surface + least-cost | [10.1038/s41591-020-1059-1](https://doi.org/10.1038/s41591-020-1059-1) |
| **Alix-Garcia, McIntosh, Sims & Welch** (2013) — *Progresa & deforestation* | REStat | MODIS + Progresa localities + roads | RD + zonal stats | [10.1162/REST_a_00349](https://doi.org/10.1162/REST_a_00349) |

### Easiest starters (course project ideas)

- **Assunção et al.** — open code + open rasters
- **Alix-Garcia et al. 2015** — full openICPSR package
- **Storeygard** — nightlights are trivial to fetch
- **Weiss et al.** — Malaria Atlas Project gives you the rasters ready-made

---
## Summary

| Task | Reach for |
|------|-----------|
| Interactive map, <10k features | `folium` |
| Interactive map, >100k features | `lonboard` |
| Read Landsat/Sentinel, compute NDVI | `geowombat` |
| Stream a raster from AWS S3 | `geowombat` + COG URL |
| Store 10 GB shapefile efficiently | `GeoParquet` |
| Spatial join in SQL | `DuckDB` spatial |
| Parallelise `sjoin` across cores | `dask-geopandas` |
| Publish a shareable HTML map | `folium.save()` |

**Next:** open `Lecture2_Tutorial.ipynb` — the same tools applied to a
single end-to-end Peru fires example.